In [32]:
### Getting the data from the internet

import requests
import pandas as pd

# Define the endpoint URL
url = "https://custom.resbank.co.za/SarbWebApi/SarbData/IFData/GetInstitutionData/BA900/2025-01-01/TOTAL"

# Set headers to mimic a browser
headers = {
    "Accept": "application/json, text/javascript, */*; q=0.01",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36",
    "Origin": "https://www.resbank.co.za",
    "Referer": "https://www.resbank.co.za/",
    "Connection": "keep-alive"
}

# Send GET request
response = requests.get(url, headers=headers)

# Check success
if response.status_code == 200:
    print("✅ Successfully fetched BA900 TOTAL data.")

    # Parse JSON
    data_json = response.json()

    # Convert to DataFrame
    df = pd.DataFrame(data_json)

    # Save to CSV
    df.to_csv("BA900_TOTAL_2025-01-01.csv", index=False)
    print("💾 CSV file saved: BA900_TOTAL_2025-01-01.csv")

    # Optional preview
    print(df.head())
else:
    print(f"❌ Failed to fetch data. Status code: {response.status_code}")


✅ Successfully fetched BA900 TOTAL data.
💾 CSV file saved: BA900_TOTAL_2025-01-01.csv
  IFType InstitutionId InstitutionName      Period  LastUpdate  \
0  BA900         TOTAL         *TOTAL*  2025-01-01  2025-04-11   

                                             XMLData  
0  <SARBForms xmlns:xsi="http://www.w3.org/2001/X...  


In [33]:
df.head()

,IFType,InstitutionId,InstitutionName,Period,LastUpdate,XMLData
0,BA900,TOTAL,*TOTAL*,2025-01-01,2025-04-11,"<SARBForms xmlns:xsi=""http://www.w3.org/2001/X..."


In [34]:
file_path = './BA900_TOTAL_2025-01-01.csv'

tenth_column_data = []

with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()[1:]  # Skip the first line (header)
    for line in lines:
        
        columns = line.strip().split(',')
        data = columns[5].replace('\"\"', '"').replace('\"<','<').replace('\">','>')
        

In [35]:
data

'<SARBForms xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:xsd="http://www.w3.org/2001/XMLSchema" Type="BA900" Description="BA900 Forms" TheYear="2025" TheMonth="01" TheDay="31" InstitutionCode="TOTAL" InstitutionDescription="*TOTAL*" LastModified="20250131>  <SARBForm>    <Table TableNumber="1" TableDescription="LIABILITIES AT MONTH-END>      <ColumnHeader ColumnNumber="1" ColumnCode="1" ColumnDescription="Chequej" />      <ColumnHeader ColumnNumber="2" ColumnCode="2" ColumnDescription="Savings" />      <ColumnHeader ColumnNumber="3" ColumnCode="3" ColumnDescription="Up to 1 day" />      <ColumnHeader ColumnNumber="4" ColumnCode="4" ColumnDescription="More than 1 day to 1 month" />      <ColumnHeader ColumnNumber="5" ColumnCode="5" ColumnDescription="More than 1 month to 6 months" />      <ColumnHeader ColumnNumber="6" ColumnCode="6" ColumnDescription="More than 6 months" />      <ColumnHeader ColumnNumber="7" ColumnCode="7" ColumnDescription="TOTAL" />      <ColumnHeader

In [36]:
import xml.etree.ElementTree as ET
import pandas as pd

# Step 2: Prepare for parsing
all_rows = []
column_names = []

# Step 3: Loop through XML and parse it
for xml_str in data:
    try:
        root = ET.fromstring(xml_str)

        # Each SARBForm in SARBForms
        for sarb_form in root.findall(".//SARBForm"):
            for table in sarb_form.findall("Table"):

                # Extract headers (once)
                if not column_names:
                    headers = table.findall("ColumnHeader")
                    column_names = [col.attrib.get("ColumnDescription", "") for col in headers]

                # Extract rows
                for row in table.findall("Row"):
                    item_number = row.attrib.get("ItemNumber")
                    item_desc = row.attrib.get("ItemDescription")
                    values = [col.attrib.get("Value") for col in row.findall("Column")]
                    all_rows.append([item_number, item_desc] + values)

    except ET.ParseError as e:
        print("Skipping malformed XML:", e)

# Step 4: Build DataFrame
if all_rows:
    column_headers = ["Item Number", "Description"] + column_names
    df = pd.DataFrame(all_rows, columns=column_headers)
    df.iloc[:, 2:] = df.iloc[:, 2:].apply(pd.to_numeric, errors="coerce")
    print(df.head())
else:
    print("No rows were parsed from the XML.")


Skipping malformed XML: unclosed token: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: no element found: line 1, column 1
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skipping malformed XML: syntax error: line 1, column 0
Skip

In [37]:
df.head()

,IFType,InstitutionId,InstitutionName,Period,LastUpdate,XMLData
0,BA900,TOTAL,*TOTAL*,2025-01-01,2025-04-11,"<SARBForms xmlns:xsi=""http://www.w3.org/2001/X..."


In [41]:
# Extract the last column
last_column_df = df[[df.columns[-1]]]

In [42]:
last_column_df

,XMLData
0,"<SARBForms xmlns:xsi=""http://www.w3.org/2001/X..."


In [44]:
import xml.etree.ElementTree as ET
import pandas as pd

# Get the XML string from the first row of the last column
xml_string = last_column_df.iloc[0, 0]  # First row, only column

# Parse the XML
root = ET.fromstring(xml_string)

# Output containers
all_rows = []
column_names = []

# Loop through <SARBForm> elements
for sarb_form in root.findall(".//SARBForm"):
    for table in sarb_form.findall("Table"):

        # Capture column headers (only once)
        if not column_names:
            headers = table.findall("ColumnHeader")
            column_names = [col.attrib.get("ColumnDescription", "") for col in headers]

        # Extract rows
        for row in table.findall("Row"):
            item_number = row.attrib.get("ItemNumber")
            item_desc = row.attrib.get("ItemDescription")
            values = [col.attrib.get("Value") for col in row.findall("Column")]
            all_rows.append([item_number, item_desc] + values)

# Assemble final DataFrame
column_headers = ["Item Number", "Description"] + column_names
df_parsed = pd.DataFrame(all_rows, columns=column_headers)

# Convert numeric columns
df_parsed.iloc[:, 2:] = df_parsed.iloc[:, 2:].apply(pd.to_numeric, errors="coerce")

# Preview
df_parsed.head()


,Item Number,Description,Chequej,Savings,Up to 1 day,More than 1 day to 1 month,More than 1 month to 6 months,More than 6 months,TOTAL,NCDs/PNs i (included in col. 7)
0,1,DEPOSITS (total of items 2 and 32),1288121307.9387,542081765.2382,1658054849.502,656246956.3666,851514385.0601,1107158452.0447,6103177716.1602,628135129.8923
1,2,DEPOSITS DENOMINATED IN RAND (total of items 3...,1223229379.5335,540712773.2382,1525373369.2173,601704431.1506,781570922.8921,1053071490.505,5725662365.5467,628135129.8923
2,3,SA banksb (total of items 4 and 5),6135273.3098,87438.5142,35709635.7576,7018227.5292,12258468.7327,34506049.9843,95715092.8177,14630478.0968
3,4,NCDs/PNsi,NaN,NaN,0.0,979368.0,3187381.9883,10463727.1085,14630477.0968,14630478.0968
4,5,Other deposits,6135273.3098,87438.5142,35709635.7576,6038859.5292,9071086.7444,24042322.8758,81084615.7209,NaN


In [45]:
import pandas as pd
import numpy as np
import requests

In [ ]:
# ### Importing data from CSV file as the webscraping is not working at this moment

# df = pd.read_csv("./TOTAL_2025-01-01.csv", skiprows=6)

In [46]:
df_parsed.head()

,Item Number,Description,Chequej,Savings,Up to 1 day,More than 1 day to 1 month,More than 1 month to 6 months,More than 6 months,TOTAL,NCDs/PNs i (included in col. 7)
0,1,DEPOSITS (total of items 2 and 32),1288121307.9387,542081765.2382,1658054849.502,656246956.3666,851514385.0601,1107158452.0447,6103177716.1602,628135129.8923
1,2,DEPOSITS DENOMINATED IN RAND (total of items 3...,1223229379.5335,540712773.2382,1525373369.2173,601704431.1506,781570922.8921,1053071490.505,5725662365.5467,628135129.8923
2,3,SA banksb (total of items 4 and 5),6135273.3098,87438.5142,35709635.7576,7018227.5292,12258468.7327,34506049.9843,95715092.8177,14630478.0968
3,4,NCDs/PNsi,NaN,NaN,0.0,979368.0,3187381.9883,10463727.1085,14630477.0968,14630478.0968
4,5,Other deposits,6135273.3098,87438.5142,35709635.7576,6038859.5292,9071086.7444,24042322.8758,81084615.7209,NaN


In [47]:
df_parsed.shape

(337, 10)

In [48]:
df_parsed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 337 entries, 0 to 336
Data columns (total 10 columns):
 #   Column                            Non-Null Count  Dtype 
---  ------                            --------------  ----- 
 0   Item Number                       337 non-null    object
 1   Description                       337 non-null    object
 2   Chequej                           325 non-null    object
 3   Savings                           314 non-null    object
 4   Up to 1 day                       271 non-null    object
 5   More than 1 day to 1 month        265 non-null    object
 6   More than 1 month to 6 months     298 non-null    object
 7   More than 6 months                225 non-null    object
 8   TOTAL                             40 non-null     object
 9   NCDs/PNs i  (included in col. 7)  39 non-null     object
dtypes: object(10)
memory usage: 26.5+ KB


In [49]:
# Step 1: Convert all financial columns to numeric (force errors to NaN)
financial_columns = df_parsed.columns[2:]  # all columns except 'Description' and 'Item Number'

# Apply conversion
df_parsed[financial_columns] = df_parsed[financial_columns].apply(pd.to_numeric, errors='coerce')

# Step 2: Drop rows where all financial columns are NaN
df_cleaned = df_parsed.dropna(subset=financial_columns, how='all').copy()

# Step 3: Fill missing descriptions or item numbers if needed
df_cleaned['Description'] = df_cleaned['Description'].fillna(method='ffill')
df_cleaned['Item Number'] = df_cleaned['Item Number'].fillna(method='ffill')


C:\Users\andre\AppData\Local\Temp\ipykernel_15928\1272318095.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_cleaned['Description'] = df_cleaned['Description'].fillna(method='ffill')
C:\Users\andre\AppData\Local\Temp\ipykernel_15928\1272318095.py:12: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_cleaned['Item Number'] = df_cleaned['Item Number'].fillna(method='ffill')


In [63]:
# Preview cleaned data
df_cleaned.head(20)

,Item Number,Description,Chequej,Savings,Up to 1 day,More than 1 day to 1 month,More than 1 month to 6 months,More than 6 months,TOTAL,NCDs/PNs i (included in col. 7)
0,1,DEPOSITS (total of items 2 and 32),1.288121e+09,5.420818e+08,1.658055e+09,6.562470e+08,8.515144e+08,1.107158e+09,6.103178e+09,6.281351e+08
1,2,DEPOSITS DENOMINATED IN RAND (total of items 3...,1.223229e+09,5.407128e+08,1.525373e+09,6.017044e+08,7.815709e+08,1.053071e+09,5.725662e+09,6.281351e+08
2,3,SA banksb (total of items 4 and 5),6.135273e+06,8.743851e+04,3.570964e+07,7.018228e+06,1.225847e+07,3.450605e+07,9.571509e+07,1.463048e+07
3,4,NCDs/PNsi,NaN,NaN,0.000000e+00,9.793680e+05,3.187382e+06,1.046373e+07,1.463048e+07,1.463048e+07
4,5,Other deposits,6.135273e+06,8.743851e+04,3.570964e+07,6.038860e+06,9.071087e+06,2.404232e+07,8.108462e+07,NaN
5,6,Central and provincial government sector depos...,1.185018e+08,3.873154e+06,9.482221e+07,2.687096e+07,3.738198e+07,3.049154e+07,3.119417e+08,6.363487e+06
6,7,Central government of the Republic (total of i...,1.066709e+08,3.069751e+06,8.984161e+07,1.853807e+07,2.233971e+07,2.396496e+07,2.644250e+08,5.759865e+06
7,8,Tax and loan account,6.819361e+07,0.000000e+00,3.341347e+07,8.166000e+03,1.147100e+04,6.406000e+03,1.016331e+08,0.000000e+00
8,9,Other,3.847733e+07,3.069751e+06,5.642814e+07,1.852990e+07,2.232824e+07,2.395856e+07,1.627919e+08,5.759865e+06
9,10,Provincial governments,1.065060e+07,7.709062e+05,4.225972e+06,7.017028e+06,1.155092e+07,3.784598e+06,3.800002e+07,6.036230e+05


In [64]:
df_cleaned.shape

(331, 10)

In [65]:
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
Index: 331 entries, 0 to 336
Data columns (total 10 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Item Number                       331 non-null    object 
 1   Description                       331 non-null    object 
 2   Chequej                           325 non-null    float64
 3   Savings                           314 non-null    float64
 4   Up to 1 day                       271 non-null    float64
 5   More than 1 day to 1 month        265 non-null    float64
 6   More than 1 month to 6 months     298 non-null    float64
 7   More than 6 months                225 non-null    float64
 8   TOTAL                             40 non-null     float64
 9   NCDs/PNs i  (included in col. 7)  39 non-null     float64
dtypes: float64(8), object(2)
memory usage: 28.4+ KB


In [66]:
df_cleaned.isnull().sum()

Item Number                           0
Description                           0
Chequej                               6
Savings                              17
Up to 1 day                          60
More than 1 day to 1 month           66
More than 1 month to 6 months        33
More than 6 months                  106
TOTAL                               291
NCDs/PNs i  (included in col. 7)    292
dtype: int64

In [67]:
df_cleaned.fillna(0, inplace=True)

In [68]:
df_cleaned.isnull().sum()

Item Number                         0
Description                         0
Chequej                             0
Savings                             0
Up to 1 day                         0
More than 1 day to 1 month          0
More than 1 month to 6 months       0
More than 6 months                  0
TOTAL                               0
NCDs/PNs i  (included in col. 7)    0
dtype: int64

In [69]:
# Step 1: Get NPL ratio using Item Numbers 110 (Loans) and 194 (Impairments)
loans_total = df_cleaned[df_cleaned["Item Number"] == "110"]["Chequej"].values[0]
impairments_total = df_cleaned[df_cleaned["Item Number"] == "194"]["Chequej"].values[0]
npl_ratio = (impairments_total / loans_total) * 100

In [70]:
npl_ratio

4.159749060823435

In [71]:
X = df_cleaned[financial_columns]
y = pd.Series([npl_ratio] * len(X), name="NPL_Ratio")

In [72]:
from sklearn.impute import SimpleImputer

# Impute all financial columns with mean
imputer = SimpleImputer(strategy="mean")
X_full = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

In [73]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [74]:
# Confirm X has data and correct type
print("X shape:", X_full.shape)

X shape: (331, 8)


In [75]:
# Fit model
model_full = LinearRegression()
model_full.fit(X_full, y)
y_pred_full = model_full.predict(X_full)

# Evaluate
mse = mean_squared_error(y, y_pred_full)
r2 = r2_score(y, y_pred_full)

# Show model output
model_full.coef_, model_full.intercept_, mse, r2


(array([ 2.60598689e-40, -1.80931335e-39,  4.04799104e-39, -7.75763161e-40,
        -7.24614017e-40, -1.49690607e-39, -8.51392554e-40,  2.82349022e-39]),
 4.159749060823436,
 7.888609052210118e-31,
 0.0)

In [77]:
# Store model summary
model_summary = {
    "coefficients": model_full.coef_,
    "intercept": model_full.intercept_,
    "mse": mse,
    "r2": r2,
    "features": X.columns.tolist()
}

model_summary_df = pd.DataFrame({
    "Feature": model_summary["features"],
    "Coefficient": model_summary["coefficients"]
})

In [78]:
# Export this to use in the Streamlit app
df_cleaned.to_csv("./cleaned_features.csv", index=False)
model_summary_df.to_csv("./model_summary.csv", index=False)